🔐 **Data Encryption Pipeline (End-to-End Secure Processing)**

This module implements a structured encryption pipeline for securely handling incoming wearable data across the system. It ensures that sensitive health information remains protected throughout its lifecycle—from ingestion to storage and retrieval.

🔹 **Core Functionality**

* Receives real-time wearable data in JSON format via WebSocket streams
* Applies encryption before any storage or processing
* Maintains a secure pipeline where raw data is never stored in plaintext

🔐 **Encryption Strategy**

* Uses symmetric encryption (Fernet) for fast and secure data protection
* Each data packet is encrypted immediately upon arrival
* Decryption is performed only when explicitly required (e.g., visualization or analysis)

⚙️ **Pipeline Architecture**

1. Data Ingestion (WebSocket)
2. Encryption Layer (Fernet-based cipher)
3. Secure Storage (ciphertext in database)
4. Controlled Decryption (API-level access)

🛡️ **Security Benefits**

* Prevents unauthorized access to sensitive health metrics
* Ensures data confidentiality even if storage is compromised
* Simulates real-world healthcare-grade secure pipelines

📈 **System Role**
This module acts as the security backbone of the application, ensuring all downstream components (analytics, dashboards, ML models) operate on protected data.

🚀 **Future Enhancements**

* Implement key rotation and secure key management (AWS KMS)
* Add user-specific encryption keys
* Integrate HTTPS/TLS for transport-layer security
* Apply role-based access control for decryption endpoints

---

✨ *In short: This module ensures that all wearable data is encrypted at every stage, forming a secure and reliable data pipeline.*


In [1]:
from fastapi import FastAPI, WebSocket
import uvicorn
from threading import Thread
from cryptography.fernet import Fernet
import json
from datetime import datetime

app = FastAPI()

# 🔐 Key Management (simulate secure key handling)
key = Fernet.generate_key()
cipher = Fernet(key)

# 📦 Secure Storage
database = []

# 🔐 Encryption Layer
def encrypt_data(data: dict):
    json_data = json.dumps(data)
    return cipher.encrypt(json_data.encode())

# 🔓 Decryption Layer
def decrypt_data(encrypted_data):
    decrypted = cipher.decrypt(encrypted_data).decode()
    return json.loads(decrypted)

# 🌐 Root
@app.get("/")
def home():
    return {"message": "Secure Pipeline Running"}

# 📡 WebSocket Ingestion
@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()

    while True:
        raw_data = await websocket.receive_text()
        parsed = json.loads(raw_data)

        # Add metadata
        parsed["timestamp"] = str(datetime.utcnow())

        # Encrypt before storage
        encrypted = encrypt_data(parsed)

        # Store securely
        database.append(encrypted)

        print("🔐 Encrypted:", encrypted)

        await websocket.send_text("Encrypted & Stored")

# 📊 Secure Retrieval
@app.get("/secure-data")
def get_secure_data():
    return {"encrypted_data": [str(d) for d in database]}

# 📊 Decrypted Retrieval (controlled)
@app.get("/decrypted-data")
def get_decrypted_data():
    decrypted_list = [decrypt_data(d) for d in database]
    return {"data": decrypted_list}

In [2]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

Thread(target=run).start()

In [6]:
!pip install pyngrok
from pyngrok import ngrok

ngrok.kill()

# Paste your ngrok authentication token here
ngrok.set_auth_token("3Bld7qqrmmQEgf5tP29blqaApYk_38WQcpAmLd45DtKje2n7f")

public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://matriarchal-boston-unprematurely.ngrok-free.dev" -> "http://localhost:8000"


In [7]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://matriarchal-boston-unprematurely.ngrok-free.dev" -> "http://localhost:8000"


In [12]:
import asyncio
import websockets
import json

async def send_data():
    # IMPORTANT: Replace YOUR_URL with the public_url obtained from ngrok.connect()
    uri = "wss://matriarchal-boston-unprematurely.ngrok-free.dev/ws"

    async with websockets.connect(uri) as websocket:
        payload = {
            "heart_rate": 85,
            "steps": 1200,
            "device_id": "wearable_001"
        }

        await websocket.send(json.dumps(payload))
        response = await websocket.recv()
        print(f"Received: {response}")

# Directly await the coroutine in Colab's environment
await send_data()

INFO:     34.12.236.33:0 - "WebSocket /ws" [accepted]
INFO:     connection open
/tmp/ipykernel_3565/3115231575.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  parsed["timestamp"] = str(datetime.utcnow())


🔐 Encrypted: b'gAAAAABpzWvaLxdohZ801DovH3nWStwlXUtFmyx8r0DZd0Y0yGEggPSr3UGzN8nhqcQZ97QVlcGZ7nHmgjN209phqmtE9is4-9H-vWtLZA__QmtkRh8_5eKBeR4JPohl836ZURGEJFrOnoh80BStzdYKtpNICaMN_RWyDMwSC2gXRJrCi_ZFFnauTbCFPSOZQBFCpKdWeXCNHh7a4oXqcPEGg0_kUmR0Kg=='
Received: Encrypted & Stored


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/websockets/websockets_impl.py", line 244, in run_asgi
    result = await self.app(self.scope, self.asgi_receive, self.asgi_send)  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 151, in __call__
    await self.

In [16]:
import requests
print(requests.get("https://matriarchal-boston-unprematurely.ngrok-free.dev/").json())

INFO:     34.12.236.33:0 - "GET / HTTP/1.1" 200 OK
{'message': 'Secure Pipeline Running'}


In [17]:
print(requests.get("https://matriarchal-boston-unprematurely.ngrok-free.dev/").json())

INFO:     34.12.236.33:0 - "GET / HTTP/1.1" 200 OK
{'message': 'Secure Pipeline Running'}
